# Cadence — Freeze-of-Gait capture plots

**▶ Runtime → Run all** (`Ctrl/Cmd+F9`) to generate every plot from a recorded capture.

It runs on an embedded real-freeze capture by default. To plot your own recording, see the note in the **Data** cell.

In [ ]:
# install the two non-default plotting libs if missing (Colab usually has them)
import importlib.util, subprocess, sys
for pkg in ['seaborn', 'plotly']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('deps ready.')


### Engine — inference + plotting (mirrors the device's `fog.dsp` + `fog.detect`)

In [ ]:

# Cadence — self-contained capture-analysis engine (mirrors fog.dsp + fog.detect).
import numpy as np, scipy.signal as signal
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
import seaborn as sns

FS, WINDOW, HOP = 64, 256, 128
LOCO_BAND, FREEZE_BAND, TREMOR_BAND = (0.5, 3.0), (3.0, 8.0), (4.0, 6.0)
FI_THRESHOLD, FLOOR_MIN = 1.815, 3000.0
DEFAULT_LABELS = ["still","walk","freeze","walk","freeze","walk","freeze","walk","still"]
LINE = {"ax":"#1E2761","ay":"#E67E22","az":"#2E7D32"}
BAND = {"still":"#ECECEC","walk":"#E1F0EF","freeze":"#FAE0DC"}
EDGE = {"still":"#9AA0A6","walk":"#1C7293","freeze":"#C0392B"}
PRETTY = {"still":"STILL","walk":"WALKING","freeze":"FREEZE"}
NAVY = "#1E2761"
sns.set_theme(style="whitegrid", context="talk")

def magnitude(w):
    m = np.linalg.norm(np.asarray(w, float), axis=1); return m - m.mean()

def band_power(s, fs, band):
    s = np.asarray(s, float); f, p = signal.welch(s, fs=fs, nperseg=min(len(s), 256))
    lo, hi = band; mask = (f >= lo) & (f < hi)
    if not mask.any(): return 0.0
    df = float(f[1]-f[0]) if len(f) > 1 else 1.0; return float(np.sum(p[mask]) * df)

def freeze_index(w):
    m = magnitude(w); return band_power(m, FS, FREEZE_BAND) / (band_power(m, FS, LOCO_BAND) + 1e-9)

def movement_energy(w):
    m = magnitude(w); return band_power(m, FS, LOCO_BAND) + band_power(m, FS, FREEZE_BAND)

def windows(accel):
    accel = np.asarray(accel, float); n = accel.shape[0]; rows = []
    for s in range(0, n - WINDOW + 1, HOP):
        w = accel[s:s+WINDOW]
        rows.append((float((s+WINDOW/2)/FS), float(freeze_index(w)), float(movement_energy(w))))
    if not rows: return np.empty(0), np.empty(0), np.empty(0)
    tc, fi, en = (np.array(c) for c in zip(*rows)); return tc, fi, en

def still_floor(en):
    en = np.asarray(en, float)
    if en.size == 0: return FLOOR_MIN
    le = np.log10(np.clip(en, 1.0, None))
    if float(le.max() - le.min()) < 1.5: return FLOOR_MIN
    hist, edges = np.histogram(le, bins=64); centers = 0.5*(edges[:-1]+edges[1:]); total = hist.sum()
    if total == 0: return FLOOR_MIN
    cw = np.cumsum(hist); cs = np.cumsum(hist*centers); best_i, best_t, best_v = 0, centers[0], -1.0
    for i in range(1, len(hist)):
        w0, w1 = cw[i-1], total-cw[i-1]
        if w0 == 0 or w1 == 0: continue
        m0 = cs[i-1]/w0; m1 = (cs[-1]-cs[i-1])/w1; v = w0*w1*(m0-m1)**2
        if v > best_v: best_v, best_t, best_i = v, centers[i], i
    if best_i <= 1 or best_i >= len(hist)-1: return FLOOR_MIN
    return max(float(10**best_t), FLOOR_MIN)

def infer(fi, en, floor):
    fi = np.asarray(fi, float); en = np.asarray(en, float)
    if fi.size == 0:
        e = np.empty(0, dtype="<U6"); return e, e.copy()
    raw = np.where(en <= floor, "still", np.where(fi > FI_THRESHOLD, "freeze", "walk"))
    committed, pend, run, out = "walk", None, 0, []
    for r in raw:
        run = run+1 if r == pend else 1; pend = r
        if run >= 2: committed = r
        out.append(committed)
    return raw, np.array(out)

def spans(tc, state):
    out = []
    if len(state) == 0: return out
    start = 0; half = HOP/FS/2
    for i in range(1, len(state)+1):
        if i == len(state) or state[i] != state[start]:
            out.append((tc[start]-half, tc[i-1]+half, state[start])); start = i
    return out

def truth_per_window(phase, labels, n):
    out = []
    for k in range(n):
        seg = np.clip(phase[k*HOP:k*HOP+WINDOW].astype(int), 0, None)
        dom = int(np.bincount(seg).argmax())
        out.append(labels[dom] if 0 <= dom < len(labels) else "?")
    return np.array(out)

def load_csv_text(text):
    labels = None; data = []; header = False
    for ln in text.splitlines():
        s = ln.strip()
        if not s: continue
        if s.startswith("#"):
            if "--labels" in s:
                tail = s.split("--labels", 1)[1].strip().split("--freeze-phases")[0]
                labs = [x.strip() for x in tail.replace(" ", ",").split(",") if x.strip()]
                if labs: labels = labs
            continue
        if not header and s.lower().startswith("idx"): header = True; continue
        p = s.split(",")
        try: rec = (int(float(p[0])), float(p[1]), float(p[2]), float(p[3]), float(p[4]), int(float(p[5])))
        except (IndexError, ValueError): continue
        if all(np.isfinite(v) for v in rec[1:5]): data.append(rec)
    arr = np.array(data, float) if data else np.empty((0, 6))
    return arr, (labels or DEFAULT_LABELS)

# ────────────────────────────── plots (use globals set in the DATA cell) ──────
def _axis_band_legend(ax):
    ah, al = ax.get_legend_handles_labels()
    bh = [mpatches.Patch(facecolor=BAND[c], edgecolor=EDGE[c], label=PRETTY[c]) for c in ["still","walk","freeze"]]
    ax.legend(ah+bh, al+[h.get_label() for h in bh], ncol=6, loc="upper center",
              bbox_to_anchor=(0.5,-0.13), frameon=False, fontsize=11)

def fig_axes(shade=True):
    cat = np.array([labels[int(p)] if 0 <= int(p) < len(labels) else "?" for p in phase])
    fig, ax = plt.subplots(figsize=(15,5))
    if shade:
        st = 0
        for i in range(1, len(cat)+1):
            if i == len(cat) or cat[i] != cat[st]:
                ax.axvspan(t[st], t[i] if i < len(cat) else t[-1], color=BAND.get(cat[st],"#fff"), zorder=0); st = i
    for nm, k in [("ax",0),("ay",1),("az",2)]:
        ax.plot(t, accel[:,k], lw=0.5, color=LINE[nm], label=nm)
    ax.set(xlabel="time (s)", ylabel="acceleration (mg)"); ax.margins(x=0.004)
    ax.set_title(f"{stem}: 3-axis acceleration" + (" by gait phase" if shade else ""), color=NAVY, fontweight="bold")
    _axis_band_legend(ax); plt.tight_layout(); plt.show()

def fig_predicted():
    fig, ax = plt.subplots(figsize=(15,5))
    for t0,t1,c in spans(tc, pred): ax.axvspan(t0, t1, color=BAND.get(c,"#fff"), zorder=0)
    for nm, k in [("ax",0),("ay",1),("az",2)]:
        ax.plot(t, accel[:,k], lw=0.5, color=LINE[nm], label=nm)
    ax.set(xlabel="time (s)", ylabel="acceleration (mg)"); ax.margins(x=0.004)
    ax.set_title(f"{stem}: what the MODEL inferred (still / walk / freeze)", color=NAVY, fontweight="bold")
    _axis_band_legend(ax); plt.tight_layout(); plt.show()

def fig_truth_vs_pred():
    agree = float((truth == pred).mean())
    tf, pf = truth == "freeze", pred == "freeze"
    tp = int((tf & pf).sum()); fn = int((tf & ~pf).sum()); fp = int((~tf & pf).sum())
    fig,(a1,a2,a3)=plt.subplots(3,1,figsize=(15,8),sharex=True,gridspec_kw={"height_ratios":[2.2,1.5,1.5]})
    for nm,k in [("ax",0),("ay",1),("az",2)]: a1.plot(t, accel[:,k], lw=0.5, color=LINE[nm])
    a1.set_ylabel("accel (mg)"); a1.margins(x=0.004)
    a1.set_title(f"{stem}: you (ground truth) vs the model  —  agreement {agree:.0%} · freeze caught {tp}/{tp+fn}",
                 color=NAVY, fontweight="bold")
    def strip(st, y, lab):
        for t0,t1,c in spans(tc, st):
            a2.add_patch(mpatches.Rectangle((t0,y), t1-t0, 0.8, color=BAND.get(c,"#fff"), ec=EDGE.get(c,"#bbb"), lw=0.2))
        a2.text(-3, y+0.4, lab, ha="right", va="center", fontsize=11, fontweight="bold")
    strip(truth, 1.15, "you\n(truth)"); strip(pred, 0.15, "model\n(pred)")
    tick = HOP/FS
    for k in np.where(truth != pred)[0]:
        a2.add_patch(mpatches.Rectangle((tc[k]-tick/2, 1.02), tick, 0.10, color="#C0392B", lw=0))
    a2.set_xlim(tc[0]-4, tc[-1]); a2.set_ylim(0,2.05); a2.set_yticks([])
    for sp in ["top","right","left"]: a2.spines[sp].set_visible(False)
    a2.text(tc[-1], 1.0, "  red = disagree", ha="right", va="center", fontsize=8, color="#C0392B")
    a3.step(tc, fi, where="mid", color="#C0392B", lw=1.1); a3.axhline(FI_THRESHOLD, ls="--", lw=1, color=NAVY)
    a3.text(tc[-1], FI_THRESHOLD, f" freeze line {FI_THRESHOLD}", va="bottom", ha="right", fontsize=9, color=NAVY)
    a3.set(yscale="log", ylabel="Freeze Index", xlabel="time (s)"); a3.grid(alpha=0.25); a3.margins(x=0.004)
    plt.tight_layout(); plt.show()

def fig_fi_timeline():
    fig, ax = plt.subplots(figsize=(15,4))
    for t0,t1,c in spans(tc, pred): ax.axvspan(t0, t1, color=BAND.get(c,"#fff"), zorder=0)
    ax.step(tc, fi, where="mid", color="#C0392B", lw=1.4)
    ax.axhline(FI_THRESHOLD, ls="--", lw=1, color=NAVY)
    ax.text(tc[-1], FI_THRESHOLD, f" freeze line {FI_THRESHOLD}", va="bottom", ha="right", fontsize=10, color=NAVY)
    ax.set(yscale="log", xlabel="time (s)", ylabel="Freeze Index"); ax.margins(x=0.004)
    ax.set_title(f"{stem}: Freeze Index over time", color=NAVY, fontweight="bold")
    plt.tight_layout(); plt.show()

def fig_freeze_zoom():
    best_i, best = -1, -1.0
    for k in range(len(tc)):
        if (truth[k] == "freeze" or pred[k] == "freeze") and fi[k] > best:
            best, best_i = fi[k], k
    if best_i < 0:
        print("No freeze window in this capture — nothing to zoom into."); return
    s = best_i*HOP; w = accel[s:s+WINDOW]
    wm = np.linalg.norm(w, axis=1); wm = wm - wm.mean(); tw = np.arange(WINDOW)/FS
    f, p = signal.welch(wm, fs=FS, nperseg=WINDOW)
    fig,(a1,a2)=plt.subplots(1,2,figsize=(15,4.4))
    a1.plot(tw, wm, color="#C0392B", lw=1.3)
    a1.set_title(f"Freeze close-up @ t~{tc[best_i]:.0f}s  (FI={best:.1f})", color=NAVY, fontweight="bold")
    a1.set(xlabel="time within window (s)", ylabel="mean-removed |accel| (mg)"); a1.grid(alpha=0.3)
    a2.semilogy(f, p+1e-9, color=NAVY, lw=1.6)
    a2.axvspan(0.5,3, color="#1C7293", alpha=0.15, label="locomotor 0.5-3 Hz")
    a2.axvspan(3,8, color="#C0392B", alpha=0.18, label="freeze 3-8 Hz")
    a2.set_xlim(0,15); a2.set_title("Power spectrum of that window", color=NAVY, fontweight="bold")
    a2.set(xlabel="frequency (Hz)", ylabel="power (log)"); a2.legend(frameon=False, fontsize=10); a2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

def fig_fi_per_phase():
    cats = [c for c in ["still","walk","freeze"] if (truth == c).any()]
    means = [float(fi[truth == c].mean()) for c in cats]
    maxs = [float(fi[truth == c].max()) for c in cats]
    x = np.arange(len(cats)); fig, ax = plt.subplots(figsize=(9,5))
    ax.bar(x-0.2, means, 0.4, color=[EDGE[c] for c in cats], label="mean FI")
    ax.bar(x+0.2, maxs, 0.4, color=[BAND[c] for c in cats], edgecolor=[EDGE[c] for c in cats], label="max FI")
    ax.axhline(FI_THRESHOLD, ls="--", color=NAVY, lw=1); ax.text(len(cats)-0.5, FI_THRESHOLD, f" {FI_THRESHOLD}", color=NAVY, fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels([PRETTY[c] for c in cats]); ax.set_ylabel("Freeze Index")
    ax.set_title(f"{stem}: Freeze Index per phase", color=NAVY, fontweight="bold"); ax.legend(frameon=False)
    plt.tight_layout(); plt.show()

def fig_interactive():
    import plotly.graph_objects as go
    fig = go.Figure()
    for t0,t1,c in spans(tc, pred):
        fig.add_vrect(x0=t0, x1=t1, fillcolor=BAND.get(c,"#fff"), opacity=0.5, layer="below", line_width=0)
    for nm,k in [("ax",0),("ay",1),("az",2)]:
        fig.add_trace(go.Scattergl(x=t, y=accel[:,k], name=nm, mode="lines", line=dict(width=1, color=LINE[nm])))
    fig.update_layout(title=f"{stem}: 3-axis acceleration, shaded by inferred state (interactive)",
                      template="plotly_white", xaxis_title="time (s)", yaxis_title="acceleration (mg)",
                      legend=dict(orientation="h", y=-0.2), hovermode="x unified", height=460)
    fig.show()
print("engine loaded.")


### Data — load the capture (edit `CAPTURE` / `CAPTURE_PATH` here)

In [ ]:
# ── DATA ──────────────────────────────────────────────────────────────────
import base64, gzip
CAPTURES = {
    "smoketest": """...""",
    "capture_20260605_103351": """...""",
}

# Pick which embedded capture to plot (smoketest has a genuine freeze):
CAPTURE = "smoketest"
# To plot YOUR OWN recording instead: upload the cpx_dump CSV via the Colab
# file panel (left sidebar) and set the path here, e.g. "/content/myrun.csv":
CAPTURE_PATH = None

if CAPTURE_PATH:
    text = open(CAPTURE_PATH).read(); stem = CAPTURE_PATH.split('/')[-1].rsplit('.',1)[0]
else:
    text = gzip.decompress(base64.b64decode(CAPTURES[CAPTURE])).decode(); stem = CAPTURE

arr, labels = load_csv_text(text)
assert arr.shape[0] >= WINDOW, 'capture too short (need >= 256 samples)'
t, accel, phase = arr[:,1], arr[:,2:5], arr[:,5].astype(int)
tc, fi, en = windows(accel)
floor = still_floor(en)
raw, committed = infer(fi, en, floor)
pred = committed                      # debounced state — matches the board
truth = truth_per_window(phase, labels, len(tc))
print(f'{stem}: {arr.shape[0]} samples, {t[-1]:.0f}s, {len(tc)} windows, still_floor={floor:.0f}')


## 1. The raw 3-axis signal

In [ ]:
fig_axes(shade=True)   # shaded by what you did (the phase column)

In [ ]:
fig_axes(shade=False)  # plain — just the three traces

## 2. What the device inferred (still / walk / freeze)

In [ ]:
fig_predicted()

## 3. Cross-examine — you (truth) vs the model

In [ ]:
fig_truth_vs_pred()

## 4. Freeze Index over time

In [ ]:
fig_fi_timeline()

## 5. A freeze up close — the 3–8 Hz signature

In [ ]:
fig_freeze_zoom()

## 6. Freeze Index per phase

In [ ]:
fig_fi_per_phase()

## 7. Interactive (hover + zoom)

In [ ]:
fig_interactive()